In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

widsworldwide_globaldathon26_path = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load Data

train = pd.read_csv('/kaggle/input/WiDSWorldWide_GlobalDathon26/train.csv')
test = pd.read_csv('/kaggle/input/WiDSWorldWide_GlobalDathon26/test.csv')

drop_cols = ['event', 'event_id', 'time_to_hit_hours']
X = train.drop(columns=drop_cols).fillna(0)
y = train['event']
X_test = test.drop(columns=['event_id']).fillna(0)

# Stronger but stable GB
gb = GradientBoostingClassifier(
    n_estimators=1600,
    learning_rate=0.018,
    max_depth=3,
    subsample=0.9,
    random_state=42
)
gb.fit(X, y)
gb_prob = gb.predict_proba(X_test)[:, 1]

# Strong but not overfit RF
rf = RandomForestClassifier(
    n_estimators=1400,
    max_depth=9,
    min_samples_leaf=2,
    random_state=42
)
rf.fit(X, y)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Slightly more GB weight (better ranking)
final_prob = 0.75 * gb_prob + 0.25 * rf_prob

# Mild sharpening for ranking separation
final_prob = np.clip(final_prob ** 0.97, 0, 1)

# Horizon shaping — tuned for 48h calibration
p12 = final_prob ** 2.00
p24 = final_prob ** 1.45
p48 = final_prob ** 1.08
p72 = final_prob

# Enforce monotonicity
p24 = np.maximum(p24, p12)
p48 = np.maximum(p48, p24)
p72 = np.maximum(p72, p48)

submission = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h': p12,
    'prob_24h': p24,
    'prob_48h': p48,
    'prob_72h': p72
})

submission.to_csv('submission.csv', index=False)